In [2]:
# !pip install s3fs zarr cftime -q &> log.log

In [28]:
import geopandas as gpd
import pandas as pd

outlets = gpd.read_file("geoglows_hydrosheds_outlets.gpkg", engine="pyogrio")
outlets.head()

,name,uparea,linkno,geometry
0,RUFIJI,1.783298e+08,110003934,"LINESTRING (36.82967 -18.19322, 36.82622 -18.1..."
1,SETIT,5.556828e+08,110015837,"LINESTRING (33.95867 26.75622, 33.95522 26.752..."
2,SETIT,1.782745e+08,110018504,"LINESTRING (34.84811 25.17900, 34.84256 25.173..."
3,SETIT,1.079791e+09,110019725,"LINESTRING (32.91433 28.56300, 32.90744 28.555..."
4,SETIT,1.949503e+08,110021123,"LINESTRING (34.74711 25.34300, 34.74256 25.339..."


In [29]:
rhine_outlets = outlets[outlets.name == 'RHINE']
rhine_outlets.head()

,name,uparea,linkno,geometry
2717,RHINE,278383712.0,230087803,"MULTILINESTRING ((-1.33747 43.89146, -1.33733 ..."
2720,RHINE,512698752.0,230088883,"LINESTRING (-1.33079 44.08979, -1.32383 44.08879)"
2725,RHINE,245355408.0,230093253,"LINESTRING (-1.44734 43.65568, -1.44024 43.65468)"
2727,RHINE,450459712.0,230094276,"LINESTRING (-0.69399 45.23512, -0.69051 45.236..."
2735,RHINE,172031616.0,230098622,"LINESTRING (-0.89925 45.41168, -0.93281 45.38934)"


In [3]:
import numpy as np
import xarray as xr

In [4]:
ds = xr.open_zarr('s3://geoglows-v2/retrospective/daily.zarr', storage_options={'anon': True})

In [5]:
ds

<xarray.Dataset>
Dimensions:   (time: 31546, river_id: 6838900)
Coordinates:
  * river_id  (river_id) int32 110229254 110230566 ... 820224554 820211113
  * time      (time) datetime64[ns] 1940-01-01 1940-01-02 ... 2026-05-14
Data variables:
    Q         (time, river_id) float32 dask.array<chunksize=(31047, 50), meta=np.ndarray>

# 01. Rhine IDs

In [8]:
rhine_ids = list(rhine_outlets.linkno)

In [12]:
rhine_ds = (
    ds
    .sel(river_id=rhine_ids)
    .sel(time=slice("2002-01-01", None)) #2002 onwards
)



# create year-month timestamps (monthly index)
month_index = pd.to_datetime(rhine_ds.time.values).to_period("M").to_timestamp()

rhine_ds = rhine_ds.assign_coords(month=("time", month_index))

rhine_ds_monthly = rhine_ds.groupby("month").mean()

# 02. All IDs

In [25]:
outlet_ids = list(outlets.linkno)

In [26]:
outlets_ds = (
    ds
    .sel(river_id=outlet_ids)
    .sel(time=slice("2002-01-01", None)) #2002 onwards
)



# create year-month timestamps (monthly index)
month_index = pd.to_datetime(outlets_ds.time.values).to_period("M").to_timestamp()

outlets_ds = outlets_ds.assign_coords(month=("time", month_index))

outlets_ds_monthly = outlets_ds.groupby("month").mean()

# 03. Export to file

In [24]:
rhine_ds_monthly.to_netcdf("rhine_monthly.nc")


In [ ]:
outlets_ds_monthly.to_netcdf("outlets_ds_monthly.nc")